<a href="https://colab.research.google.com/github/merugumallarajasimha/deep_learning_basics/blob/main/ann_with_KerasTuner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Keras Tuner ANN Pipeline

## 1. Generate Synthetic Dataset

To demonstrate the Keras Tuner, we'll create a simple binary classification dataset using `sklearn.datasets.make_classification`.

In [12]:
!pip install keras-tuner

In [13]:
!pip install -q keras-tuner

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import keras_tuner as kt
from sklearn.model_selection import train_test_split
from sklearn.datasets import make_classification
from sklearn.preprocessing import StandardScaler

X, y = make_classification(
    n_samples=1000,
    n_features=20,
    n_informative=10,
    n_redundant=5,
    n_repeated=2,
    n_classes=2,
    random_state=42
)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Shape of X_train: {X_train_scaled.shape}")
print(f"Shape of y_train: {y_train.shape}")
print(f"Shape of X_test: {X_test_scaled.shape}")
print(f"Shape of y_test: {y_test.shape}")

Shape of X_train: (800, 20)
Shape of y_train: (800,)
Shape of X_test: (200, 20)
Shape of y_test: (200,)


## 2. Define the Hypermodel (ANN Architecture)

We'll define a `build_model` function that Keras Tuner will use to create and tune the ANN. This function will include hyperparameters for the number of hidden layers, units per layer, activation function, and learning rate.

In [14]:
def build_model(hp):
    model = keras.Sequential()
    model.add(layers.Input(shape=(X_train_scaled.shape[1],)))

    for i in range(hp.Int('num_hidden_layers', 1, 3)):
        model.add(layers.Dense(
            units=hp.Int(f'units_layer_{i}', min_value=32, max_value=512, step=32),
            activation=hp.Choice(f'activation_layer_{i}', ['relu', 'tanh', 'sigmoid'])
        ))
        model.add(layers.Dropout(hp.Float(f'dropout_{i}', min_value=0.0, max_value=0.5, step=0.1)))

    model.add(layers.Dense(1, activation='sigmoid'))

    learning_rate = hp.Choice('learning_rate', values=[1e-2, 1e-3, 1e-4])
    optimizer = keras.optimizers.Adam(learning_rate=learning_rate)

    model.compile(
        optimizer=optimizer,
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

tuner = kt.Hyperband(
    build_model,
    objective='val_accuracy',
    max_epochs=10,
    factor=3,
    directory='keras_tuner_dir',
    project_name='ann_tuning'
)

print("Keras Tuner Hyperband tuner initialized.")

Reloading Tuner from keras_tuner_dir/ann_tuning/tuner0.json
Keras Tuner Hyperband tuner initialized.


## 3. Run the Hyperparameter Search

Now we'll run the search process. This will train multiple models with different hyperparameters combinations and select the best one based on the validation accuracy.

In [15]:
stop_early = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3)

tuner.search(X_train_scaled, y_train, epochs=50, validation_split=0.2, callbacks=[stop_early])

best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]

print("\nOptimal Hyperparameters:")
for param, value in best_hps.values.items():
    print(f"  {param}: {value}")


Optimal Hyperparameters:
  num_hidden_layers: 2
  units_layer_0: 224
  activation_layer_0: relu
  dropout_0: 0.2
  learning_rate: 0.001
  units_layer_1: 160
  activation_layer_1: relu
  dropout_1: 0.2
  units_layer_2: 32
  activation_layer_2: tanh
  dropout_2: 0.4
  tuner/epochs: 10
  tuner/initial_epoch: 4
  tuner/bracket: 1
  tuner/round: 1
  tuner/trial_id: 0021


## 4. Build and Evaluate the Best Model

After the search, we'll retrieve the best model found by Keras Tuner and evaluate its performance on the test set.

In [16]:
best_model = tuner.get_best_models(num_models=1)[0]

best_model.fit(X_train_scaled, y_train, epochs=50, validation_split=0.2, callbacks=[stop_early])

loss, accuracy = best_model.evaluate(X_test_scaled, y_test)
print(f"\nTest Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")

best_model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 14 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Epoch 1/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 7s 41ms/step - accuracy: 0.9641 - loss: 0.1245 - val_accuracy: 0.9438 - val_loss: 0.1603
Epoch 2/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - accuracy: 0.9563 - loss: 0.1153 - val_accuracy: 0.9312 - val_loss: 0.1782
Epoch 3/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.9641 - loss: 0.1080 - val_accuracy: 0.9312 - val_loss: 0.1666
Epoch 4/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - accuracy: 0.9750 - loss: 0.0945 - val_accuracy: 0.9375 - val_loss: 0.1654
7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - accuracy: 0.9050 - loss: 0.2698

Test Loss: 0.2698
Test Accuracy: 0.9050


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 224)            │         4,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 224)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 160)            │        36,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 160)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │           161 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 122,597 (478.90 KB)

 Trainable params: 40,865 (159.63 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 81,732 (319.27 KB)

## 5. Pipeline Integration (Conceptual)

This entire process, from data generation/loading, preprocessing, hyperparameter tuning, to model evaluation, can be encapsulated within a pipeline. For real-world use, you would replace the synthetic data generation with your actual dataset loading and preprocessing steps. The `StandardScaler` is already a part of a typical data preprocessing pipeline.

Here's a conceptual outline of how you might integrate this into a `sklearn.pipeline.Pipeline` or a custom TensorFlow Data Pipeline:

In [17]:
from sklearn.pipeline import Pipeline


def create_ann_pipeline(hp):
    model = keras.Sequential()
    model.add(layers.Input(shape=(X_train_scaled.shape[1],)))

    for i in range(hp.Int('num_hidden_layers', 1, 3)):
        model.add(layers.Dense(
            units=hp.Int(f'units_layer_{i}', min_value=32, max_value=512, step=32),
            activation=hp.Choice(f'activation_layer_{i}', ['relu', 'tanh', 'sigmoid'])
        ))
        model.add(layers.Dropout(hp.Float(f'dropout_{i}', min_value=0.0, max_value=0.5, step=0.1)))
    model.add(layers.Dense(1, activation='sigmoid'))

    learning_rate = hp.Choice('learning_rate', values=[1e-2, 1e-3, 1e-4])
    optimizer = keras.optimizers.Adam(learning_rate=learning_rate)

    model.compile(
        optimizer=optimizer,
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

print("Pipeline concept demonstrated: Data -> Preprocessing -> Hyperparameter Tuning -> Model Training -> Evaluation.")
print("To use your 'internal dataset', you would replace the 'make_classification' and 'StandardScaler' with your specific data loading and preprocessing steps.")

Pipeline concept demonstrated: Data -> Preprocessing -> Hyperparameter Tuning -> Model Training -> Evaluation.
To use your 'internal dataset', you would replace the 'make_classification' and 'StandardScaler' with your specific data loading and preprocessing steps.
